In [8]:
import pandas as pd
import numpy as np
import commons as c
import plotly.express as px
import plotly.subplots as sp
import plotly.graph_objects as go

# Get datasets

In [9]:
csv_normal_path = 'results/dataframes/results_normal.csv'
df_normal = pd.read_csv(csv_normal_path, dtype=c.type_dict)
df_normal['nature'] = 'non-equivalent'

csv_equiv_path = 'results/dataframes/results_equiv.csv'
df_equiv = pd.read_csv(csv_equiv_path, dtype=c.type_dict)
df_equiv['nature'] = 'equivalent'


In [10]:
df = pd.concat([df_equiv, df_normal], ignore_index=True)
df['true_label'] = np.where(df['true_label'] == True, "non-equivalent", "equivalent")
df['predicted_label'] = np.where(df['predicted_label'] == True, "non-equivalent", "equivalent")

# Get Box Plots

In [31]:
def print_box_plot(df, cat, file_name, type):

    df = df.copy()  # Ensure it's a copy
    
    cat_range = sorted(df[cat].unique())  # Extract unique values from the column    
    cat_mapping = {category: idx for idx, category in enumerate(cat_range)}
    df['cat_numeric'] = df[cat].map(cat_mapping)  # Map categories to numeric values

    # Add an offset to the x-axis values based on 'true_label' to spread the points
    label_offsets = {
        "equivalent": -0.1,   # Adjust this value as needed
        "non-equivalent": 0.1  # Adjust this value as needed
    }
    
    # Apply the offset to a new column for the x-axis
    df['x_offset'] = df['cat_numeric'] + df['nature'].map(label_offsets)

    label_mapping = {
        "equivalent": "Equivalent mutant",
        "non-equivalent": "Non-Equivalent mutant"
    }    
    df['nature'] = df['nature'].map(label_mapping)
    
    # Create the scatter plot with the adjusted x-axis values
    fig = px.box(#scatter
        df, 
        y=type, 
        x="x_offset", 
        color="nature", 
        category_orders={cat: cat_range},
        points=False
    ) 
    
    # Adjust layout for better visualization
    fig.update_layout(
        scattermode="group",
        xaxis=dict(
            title=cat, 
            categoryorder="array", 
            categoryarray=cat_range,
            tickvals=list(range(len(cat_range))),
            ticktext=cat_range
        ),
        yaxis_title="Distance between original and mutant",
        xaxis_title="Characteristic",
        legend_title_text="Expected value",
        boxgroupgap=0, 
        boxgap=0
    )
    
    # Save the figure
    c.setup_layout_and_save(fig, "Visualisation of the mutant detectability score for each metric", f'results/TEST_NEW/', file_name, yaxis_range=[0, 1])


In [32]:
threshold = "N"
columns = ['Qubits_number', 'gates', 'depth', 'singlequbit_gates', 'multiqubit_gates'] 


df_hw = df[df['hardware'] == 'kyiv']
# df_metric = df_hw[df_hw['metric'] == m]
# df_threshold = df_hw[df_hw['threshold'] == threshold]
    
 #for key, categories in c.table_data.items():
selected_columns = df_hw[["metric", 'nature', 'ideal_distance']]
file_name = f'visu_ideal'
print_box_plot(selected_columns, "metric", file_name, 'ideal_distance')

for hw in c.hardware:
    df_hw = df[df['hardware'] == hw]
    # df_metric = df_hw[df_hw['metric'] == m]
    # df_threshold = df_hw[df_hw['threshold'] == threshold]
    
    #for key, categories in c.table_data.items():
    selected_columns = df_hw[["metric", 'nature', 'noisy_distance']]
    file_name = f'visu_{hw}'
    print_box_plot(selected_columns, "metric", file_name, 'noisy_distance')
        